# 19. 12-way Generator Attribution — Frozen MERT + Logistic Regression

## 목적

18/18-B에서 handcrafted feature + RBF-SVM으로 확인한
**generator-specific acoustic signature**를
사전학습 음악 representation인 **MERT-v1-95M**에서도 재검증한다.

이번 실험에서는 17번에서 이미 추출한 MERT embedding cache를 그대로 재사용한다.

### Experiment A — Full FAKE
- 모든 FAKE segment
- 12-way generator classification
- 13개 MERT layer 각각 평가
- Validation Track Macro-F1 기준 best layer 선택

### Experiment B — Strict Balanced Controlled
- 18-B에서 선택한 `original_audio × generator`당 정확히 1 track
- 동일 source set
- generator별 완전 균형
- 각 track의 segment MERT embedding을 평균해 768-D track vector 생성
- Validation Macro-F1 기준 best layer 선택

### 비교
- Handcrafted + RBF-SVM
- Frozen MERT + Logistic Regression

## 1. 라이브러리 / 경로 / 입력 QC

In [ ]:
from pathlib import Path
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)

PROJECT_ROOT = Path(
    "/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project"
)

SEGMENT_PATH = (
    PROJECT_ROOT
    / "data/metadata/segment_manifest_10s.csv"
)

MERT_EMBED_PATH = (
    PROJECT_ROOT
    / "data/processed/mert/mert95m_v2_layers_float16.npy"
)

MERT_DONE_PATH = (
    PROJECT_ROOT
    / "data/processed/mert/mert95m_v2_done.npy"
)

STRICT_TRACKS_PATH = (
    PROJECT_ROOT
    / "results/generator_attribution/balanced_controlled/strict_selected_tracks.csv"
)

HANDCRAFTED_FULL_METRICS = (
    PROJECT_ROOT
    / "results/generator_attribution/handcrafted/generator_attribution_metrics.csv"
)

HANDCRAFTED_STRICT_METRICS = (
    PROJECT_ROOT
    / "results/generator_attribution/balanced_controlled/strict_balanced_metrics.csv"
)

RESULT_DIR = (
    PROJECT_ROOT
    / "results/generator_attribution/mert"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints/generator_attribution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

for p in [
    SEGMENT_PATH,
    MERT_EMBED_PATH,
    MERT_DONE_PATH,
    STRICT_TRACKS_PATH,
]:
    print(p.name, "->", p.exists())
    if not p.exists():
        raise FileNotFoundError(p)

segments = pd.read_csv(SEGMENT_PATH).reset_index(drop=True)
mert_cache = np.load(MERT_EMBED_PATH, mmap_mode="r")
done = np.load(MERT_DONE_PATH)
strict_selected_tracks = pd.read_csv(STRICT_TRACKS_PATH)

print("\nSegments   :", len(segments))
print("MERT cache :", mert_cache.shape)
print("Done       :", int(done.sum()), "/", len(done))
print("Strict tracks:", len(strict_selected_tracks))

assert len(segments) == 10077
assert mert_cache.shape == (10077, 13, 768)
assert bool(done.all())
assert len(strict_selected_tracks) == 1428

print("Input QC PASS: True")

## 2. FAKE subset / generator / leakage QC

In [ ]:
segments["cache_index"] = np.arange(len(segments))

fake = segments[
    segments["label"] == "FAKE"
].copy()

generator_order = sorted(
    fake["generator"].dropna().unique().tolist()
)

encoder = LabelEncoder()
encoder.fit(generator_order)

print("FAKE segments:", len(fake))
print("Generators:", len(generator_order))
print(generator_order)

groups = {
    s: set(
        fake.loc[
            fake["split"] == s,
            "original_audio"
        ]
    )
    for s in ["train", "val", "test"]
}

print("Train∩Val :", len(groups["train"] & groups["val"]))
print("Train∩Test:", len(groups["train"] & groups["test"]))
print("Val∩Test  :", len(groups["val"] & groups["test"]))

assert len(fake) == 9189
assert len(generator_order) == 12
assert len(groups["train"] & groups["test"]) == 0

print("MERT Generator Split QC PASS: True")

## 3. 평가 / Track aggregation 함수

In [ ]:
def top_k_accuracy(y_true, score_matrix, k=3):
    topk = np.argsort(score_matrix, axis=1)[:, -k:]
    return float(
        np.mean([
            int(y) in row
            for y, row in zip(y_true, topk)
        ])
    )


def macro_ovr_auc(y_true, score_matrix, n_classes):
    aucs = []

    for cls in range(n_classes):
        y_bin = (
            np.asarray(y_true) == cls
        ).astype(int)

        if y_bin.min() == y_bin.max():
            continue

        aucs.append(
            roc_auc_score(
                y_bin,
                score_matrix[:, cls],
            )
        )

    return float(np.mean(aucs))


def evaluate_multiclass(y_true, score_matrix, n_classes):
    pred = np.argmax(score_matrix, axis=1)

    return {
        "accuracy": float(
            accuracy_score(y_true, pred)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, pred)
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                pred,
                average="macro",
                zero_division=0,
            )
        ),
        "top3_accuracy": float(
            top_k_accuracy(
                y_true,
                score_matrix,
                3,
            )
        ),
        "macro_ovr_auc": float(
            macro_ovr_auc(
                y_true,
                score_matrix,
                n_classes,
            )
        ),
    }


def aggregate_track_probs(meta_df, probs, class_names):
    temp = meta_df[
        [
            "track_sample_id",
            "original_audio",
            "generator",
            "genre",
            "split",
        ]
    ].reset_index(drop=True).copy()

    prob_cols = []

    for j, cls in enumerate(class_names):
        col = f"prob__{cls}"
        prob_cols.append(col)
        temp[col] = probs[:, j]

    agg = {
        "original_audio": "first",
        "generator": "first",
        "genre": "first",
        "split": "first",
    }
    agg.update({
        c: "mean"
        for c in prob_cols
    })

    track = (
        temp
        .groupby(
            "track_sample_id",
            as_index=False,
        )
        .agg(agg)
    )

    return track, prob_cols

## 4. Experiment A — Full FAKE: 13개 MERT layer 탐색

각 layer마다:
- Train-only StandardScaler
- balanced Logistic Regression
- Validation segment / track 평가
- **Validation Track Macro-F1 최고 layer 선택**

Test는 layer 선택에 사용하지 않는다.

In [ ]:
full_train = fake[
    fake["split"] == "train"
].reset_index(drop=True)

full_val = fake[
    fake["split"] == "val"
].reset_index(drop=True)

full_test = fake[
    fake["split"] == "test"
].reset_index(drop=True)

train_idx = full_train["cache_index"].to_numpy()
val_idx = full_val["cache_index"].to_numpy()
test_idx = full_test["cache_index"].to_numpy()

y_train = encoder.transform(
    full_train["generator"]
)
y_val = encoder.transform(
    full_val["generator"]
)
y_test = encoder.transform(
    full_test["generator"]
)

layer_rows = []
full_artifacts = {}

for layer in range(13):
    X_train = np.asarray(
        mert_cache[train_idx, layer, :],
        dtype=np.float32,
    )
    X_val = np.asarray(
        mert_cache[val_idx, layer, :],
        dtype=np.float32,
    )

    scaler = StandardScaler()
    X_train_z = scaler.fit_transform(X_train)
    X_val_z = scaler.transform(X_val)

    clf = LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )

    clf.fit(
        X_train_z,
        y_train,
    )

    val_probs = clf.predict_proba(
        X_val_z
    )

    seg_metrics = evaluate_multiclass(
        y_val,
        val_probs,
        12,
    )

    val_track, prob_cols = aggregate_track_probs(
        full_val,
        val_probs,
        encoder.classes_,
    )

    y_val_track = encoder.transform(
        val_track["generator"]
    )

    track_metrics = evaluate_multiclass(
        y_val_track,
        val_track[prob_cols].to_numpy(),
        12,
    )

    row = {
        "layer": layer,
        "val_segment_accuracy":
            seg_metrics["accuracy"],
        "val_segment_macro_f1":
            seg_metrics["macro_f1"],
        "val_track_accuracy":
            track_metrics["accuracy"],
        "val_track_balanced_accuracy":
            track_metrics["balanced_accuracy"],
        "val_track_macro_f1":
            track_metrics["macro_f1"],
        "val_track_top3":
            track_metrics["top3_accuracy"],
        "val_track_macro_ovr_auc":
            track_metrics["macro_ovr_auc"],
    }

    layer_rows.append(row)
    full_artifacts[layer] = {
        "scaler": scaler,
        "clf": clf,
    }

    print(
        f"layer {layer:02d} | "
        f"Val Track Macro-F1="
        f"{row['val_track_macro_f1']:.4f} | "
        f"Acc="
        f"{row['val_track_accuracy']:.4f}"
    )

full_layer_search = pd.DataFrame(
    layer_rows
)

full_ranked = (
    full_layer_search
    .sort_values(
        [
            "val_track_macro_f1",
            "val_track_balanced_accuracy",
            "val_track_macro_ovr_auc",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    full_ranked.round(4)
)

BEST_FULL_LAYER = int(
    full_ranked.iloc[0]["layer"]
)

print(
    "Best Full MERT layer:",
    BEST_FULL_LAYER,
)

## 5. Full FAKE — Best layer Test 평가

In [ ]:
full_scaler = full_artifacts[
    BEST_FULL_LAYER
]["scaler"]

full_clf = full_artifacts[
    BEST_FULL_LAYER
]["clf"]

X_val = np.asarray(
    mert_cache[
        val_idx,
        BEST_FULL_LAYER,
        :
    ],
    dtype=np.float32,
)

X_test = np.asarray(
    mert_cache[
        test_idx,
        BEST_FULL_LAYER,
        :
    ],
    dtype=np.float32,
)

val_probs = full_clf.predict_proba(
    full_scaler.transform(X_val)
)

test_probs = full_clf.predict_proba(
    full_scaler.transform(X_test)
)

segment_val_metrics = evaluate_multiclass(
    y_val,
    val_probs,
    12,
)

segment_test_metrics = evaluate_multiclass(
    y_test,
    test_probs,
    12,
)

val_track, prob_cols = aggregate_track_probs(
    full_val,
    val_probs,
    encoder.classes_,
)

test_track, _ = aggregate_track_probs(
    full_test,
    test_probs,
    encoder.classes_,
)

y_val_track = encoder.transform(
    val_track["generator"]
)

y_test_track = encoder.transform(
    test_track["generator"]
)

track_val_metrics = evaluate_multiclass(
    y_val_track,
    val_track[prob_cols].to_numpy(),
    12,
)

track_test_metrics = evaluate_multiclass(
    y_test_track,
    test_track[prob_cols].to_numpy(),
    12,
)

full_metrics = pd.DataFrame([
    {
        "representation": "MERT95M",
        "experiment": "full",
        "level": "segment",
        "split": "val",
        "best_layer": BEST_FULL_LAYER,
        **segment_val_metrics,
    },
    {
        "representation": "MERT95M",
        "experiment": "full",
        "level": "segment",
        "split": "test",
        "best_layer": BEST_FULL_LAYER,
        **segment_test_metrics,
    },
    {
        "representation": "MERT95M",
        "experiment": "full",
        "level": "track",
        "split": "val",
        "best_layer": BEST_FULL_LAYER,
        **track_val_metrics,
    },
    {
        "representation": "MERT95M",
        "experiment": "full",
        "level": "track",
        "split": "test",
        "best_layer": BEST_FULL_LAYER,
        **track_test_metrics,
    },
])

display(
    full_metrics.round(4)
)

## 6. Full Test — generator별 MERT 성능 / confusion matrix

In [ ]:
full_test_scores = test_track[
    prob_cols
].to_numpy()

full_pred_int = np.argmax(
    full_test_scores,
    axis=1,
)

full_pred = encoder.inverse_transform(
    full_pred_int
)

full_report = classification_report(
    test_track["generator"],
    full_pred,
    labels=encoder.classes_,
    output_dict=True,
    zero_division=0,
)

full_per_generator = (
    pd.DataFrame(full_report)
    .T
    .loc[
        encoder.classes_,
        [
            "precision",
            "recall",
            "f1-score",
            "support",
        ]
    ]
    .reset_index()
    .rename(
        columns={
            "index": "generator"
        }
    )
)

display(
    full_per_generator
    .sort_values(
        "f1-score",
        ascending=False,
    )
    .round(4)
)

full_cm = confusion_matrix(
    test_track["generator"],
    full_pred,
    labels=encoder.classes_,
    normalize="true",
)

fig, ax = plt.subplots(
    figsize=(11, 10)
)

ConfusionMatrixDisplay(
    full_cm,
    display_labels=encoder.classes_,
).plot(
    ax=ax,
    xticks_rotation=45,
    values_format=".2f",
)

ax.set_title(
    "MERT 12-way Generator Attribution — Full Track Test"
)

fig.tight_layout()

fig.savefig(
    RESULT_DIR
    / "mert_full_track_confusion_matrix.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()

## 7. Experiment B — Strict Balanced Track MERT dataset 생성

18-B에서 선택한 정확히 1개의 `original_audio × generator` track만 사용한다.

각 track에 속한 segment의 MERT embedding을 평균해:
- track당 1개 vector
- layer당 768-D
로 만든다.

In [ ]:
strict_ids = set(
    strict_selected_tracks[
        "track_sample_id"
    ]
)

strict_segments = fake[
    fake["track_sample_id"].isin(
        strict_ids
    )
].copy()

strict_track_rows = []

for track_id, group in strict_segments.groupby(
    "track_sample_id"
):
    cache_indices = (
        group["cache_index"]
        .to_numpy()
        .astype(int)
    )

    # [segments, 13, 768] -> [13, 768]
    track_emb = np.asarray(
        mert_cache[
            cache_indices,
            :,
            :
        ],
        dtype=np.float32,
    ).mean(axis=0)

    first = group.iloc[0]

    strict_track_rows.append({
        "track_sample_id": track_id,
        "original_audio":
            first["original_audio"],
        "generator":
            first["generator"],
        "genre":
            first["genre"],
        "split":
            first["split"],
        "embedding":
            track_emb,
    })

strict_track_df = pd.DataFrame(
    strict_track_rows
)

print("Strict MERT track rows:", len(strict_track_df))

print("\nDistribution")
display(
    pd.crosstab(
        strict_track_df["generator"],
        strict_track_df["split"],
    )
)

assert len(strict_track_df) == 1428

dist = pd.crosstab(
    strict_track_df["generator"],
    strict_track_df["split"],
)

assert dist["train"].nunique() == 1
assert dist["val"].nunique() == 1
assert dist["test"].nunique() == 1
assert int(dist["train"].iloc[0]) == 83
assert int(dist["val"].iloc[0]) == 17
assert int(dist["test"].iloc[0]) == 19

print(
    "Strict MERT Track Dataset QC PASS: True"
)

## 8. Strict Balanced — 13개 MERT layer 탐색

In [ ]:
strict_train = strict_track_df[
    strict_track_df["split"] == "train"
].reset_index(drop=True)

strict_val = strict_track_df[
    strict_track_df["split"] == "val"
].reset_index(drop=True)

strict_test = strict_track_df[
    strict_track_df["split"] == "test"
].reset_index(drop=True)

sy_train = encoder.transform(
    strict_train["generator"]
)

sy_val = encoder.transform(
    strict_val["generator"]
)

sy_test = encoder.transform(
    strict_test["generator"]
)

strict_layer_rows = []
strict_artifacts = {}

for layer in range(13):
    SX_train = np.stack([
        emb[layer]
        for emb in strict_train["embedding"]
    ]).astype(np.float32)

    SX_val = np.stack([
        emb[layer]
        for emb in strict_val["embedding"]
    ]).astype(np.float32)

    scaler = StandardScaler()
    SX_train_z = scaler.fit_transform(
        SX_train
    )
    SX_val_z = scaler.transform(
        SX_val
    )

    clf = LogisticRegression(
        class_weight=None,
        max_iter=3000,
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )

    clf.fit(
        SX_train_z,
        sy_train,
    )

    val_probs = clf.predict_proba(
        SX_val_z
    )

    m = evaluate_multiclass(
        sy_val,
        val_probs,
        12,
    )

    row = {
        "layer": layer,
        "val_accuracy":
            m["accuracy"],
        "val_balanced_accuracy":
            m["balanced_accuracy"],
        "val_macro_f1":
            m["macro_f1"],
        "val_top3":
            m["top3_accuracy"],
        "val_macro_ovr_auc":
            m["macro_ovr_auc"],
    }

    strict_layer_rows.append(
        row
    )

    strict_artifacts[layer] = {
        "scaler": scaler,
        "clf": clf,
    }

    print(
        f"layer {layer:02d} | "
        f"Val Macro-F1="
        f"{row['val_macro_f1']:.4f} | "
        f"Acc="
        f"{row['val_accuracy']:.4f}"
    )

strict_layer_search = pd.DataFrame(
    strict_layer_rows
)

strict_ranked = (
    strict_layer_search
    .sort_values(
        [
            "val_macro_f1",
            "val_balanced_accuracy",
            "val_macro_ovr_auc",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    strict_ranked.round(4)
)

BEST_STRICT_LAYER = int(
    strict_ranked.iloc[0]["layer"]
)

print(
    "Best Strict MERT layer:",
    BEST_STRICT_LAYER,
)

## 9. Strict Balanced — Best layer Test

In [ ]:
strict_scaler = strict_artifacts[
    BEST_STRICT_LAYER
]["scaler"]

strict_clf = strict_artifacts[
    BEST_STRICT_LAYER
]["clf"]

SX_val = np.stack([
    emb[BEST_STRICT_LAYER]
    for emb in strict_val["embedding"]
]).astype(np.float32)

SX_test = np.stack([
    emb[BEST_STRICT_LAYER]
    for emb in strict_test["embedding"]
]).astype(np.float32)

strict_val_probs = strict_clf.predict_proba(
    strict_scaler.transform(
        SX_val
    )
)

strict_test_probs = strict_clf.predict_proba(
    strict_scaler.transform(
        SX_test
    )
)

strict_val_metrics = evaluate_multiclass(
    sy_val,
    strict_val_probs,
    12,
)

strict_test_metrics = evaluate_multiclass(
    sy_test,
    strict_test_probs,
    12,
)

strict_metrics = pd.DataFrame([
    {
        "representation": "MERT95M",
        "experiment":
            "strict_balanced_controlled",
        "split": "val",
        "best_layer":
            BEST_STRICT_LAYER,
        **strict_val_metrics,
    },
    {
        "representation": "MERT95M",
        "experiment":
            "strict_balanced_controlled",
        "split": "test",
        "best_layer":
            BEST_STRICT_LAYER,
        **strict_test_metrics,
    },
])

display(
    strict_metrics.round(4)
)

print(
    "Chance Top-1:",
    round(1/12, 4),
)

## 10. Handcrafted vs MERT — Full / Strict 비교

In [ ]:
if not HANDCRAFTED_FULL_METRICS.exists():
    raise FileNotFoundError(
        HANDCRAFTED_FULL_METRICS
    )

if not HANDCRAFTED_STRICT_METRICS.exists():
    raise FileNotFoundError(
        HANDCRAFTED_STRICT_METRICS
    )

h_full = pd.read_csv(
    HANDCRAFTED_FULL_METRICS
)

h_strict = pd.read_csv(
    HANDCRAFTED_STRICT_METRICS
)

h_full_test = h_full[
    (h_full["experiment"] == "full")
    & (h_full["level"] == "track")
    & (h_full["split"] == "test")
].copy()

h_full_test["representation"] = (
    "Handcrafted+RBF-SVM"
)

h_strict_test = h_strict[
    h_strict["split"] == "test"
].copy()

h_strict_test["representation"] = (
    "Handcrafted+RBF-SVM"
)

m_full_test = full_metrics[
    (full_metrics["level"] == "track")
    & (full_metrics["split"] == "test")
].copy()

m_full_test["representation"] = (
    "MERT95M+LR"
)

m_strict_test = strict_metrics[
    strict_metrics["split"] == "test"
].copy()

m_strict_test["representation"] = (
    "MERT95M+LR"
)

compare_cols = [
    "representation",
    "experiment",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "top3_accuracy",
    "macro_ovr_auc",
]

comparison = pd.concat(
    [
        h_full_test[compare_cols],
        m_full_test[compare_cols],
        h_strict_test[compare_cols],
        m_strict_test[compare_cols],
    ],
    ignore_index=True,
)

display(
    comparison.round(4)
)

## 11. 저장 / 최종 QC

In [ ]:
full_layer_search.to_csv(
    RESULT_DIR
    / "mert_full_layer_search.csv",
    index=False,
    encoding="utf-8-sig",
)

strict_layer_search.to_csv(
    RESULT_DIR
    / "mert_strict_layer_search.csv",
    index=False,
    encoding="utf-8-sig",
)

full_metrics.to_csv(
    RESULT_DIR
    / "mert_full_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

strict_metrics.to_csv(
    RESULT_DIR
    / "mert_strict_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

full_per_generator.to_csv(
    RESULT_DIR
    / "mert_full_per_generator_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

comparison.to_csv(
    RESULT_DIR
    / "handcrafted_vs_mert_generator_attribution.csv",
    index=False,
    encoding="utf-8-sig",
)

full_artifact = (
    CHECKPOINT_DIR
    / "generator_attribution_mert_full_lr.joblib"
)

strict_artifact = (
    CHECKPOINT_DIR
    / "generator_attribution_mert_strict_lr.joblib"
)

joblib.dump(
    {
        "classes":
            encoder.classes_.tolist(),
        "best_layer":
            BEST_FULL_LAYER,
        "scaler":
            full_scaler,
        "classifier":
            full_clf,
    },
    full_artifact,
)

joblib.dump(
    {
        "classes":
            encoder.classes_.tolist(),
        "best_layer":
            BEST_STRICT_LAYER,
        "scaler":
            strict_scaler,
        "classifier":
            strict_clf,
    },
    strict_artifact,
)

full_test_row = full_metrics[
    (full_metrics["level"] == "track")
    & (full_metrics["split"] == "test")
].iloc[0]

strict_test_row = strict_metrics[
    strict_metrics["split"] == "test"
].iloc[0]

qc = pd.DataFrame({
    "check": [
        "mert_cache_complete",
        "fake_segments",
        "generator_count",
        "full_best_layer_valid",
        "strict_track_rows",
        "strict_best_layer_valid",
        "finite_full_macro_f1",
        "finite_strict_macro_f1",
        "full_artifact_exists",
        "strict_artifact_exists",
    ],
    "value": [
        bool(done.all()),
        len(fake),
        len(generator_order),
        0 <= BEST_FULL_LAYER <= 12,
        len(strict_track_df),
        0 <= BEST_STRICT_LAYER <= 12,
        np.isfinite(
            full_test_row["macro_f1"]
        ),
        np.isfinite(
            strict_test_row["macro_f1"]
        ),
        full_artifact.exists(),
        strict_artifact.exists(),
    ],
})

display(qc)

core_qc_pass = (
    bool(done.all())
    and len(fake) == 9189
    and len(generator_order) == 12
    and 0 <= BEST_FULL_LAYER <= 12
    and len(strict_track_df) == 1428
    and 0 <= BEST_STRICT_LAYER <= 12
    and np.isfinite(
        full_test_row["macro_f1"]
    )
    and np.isfinite(
        strict_test_row["macro_f1"]
    )
    and full_artifact.exists()
    and strict_artifact.exists()
)

print("===== FINAL RESULT =====")
print(
    "MERT Generator Attribution Core QC PASS:",
    core_qc_pass,
)

## 최신 실행 결과 요약 (2026-09-13)

- Validation Track Macro-F1 기준 최적 MERT layer는 Full과 Strict 모두 **Layer 10**이다.
- Full 12-way Track Test Accuracy는 **0.9393**, Macro-F1은 **0.9384**, Macro OVR AUC는 **0.9959**다.
- Strict balanced Test Accuracy는 **0.8991**, Macro-F1은 **0.8980**이다.
- Strict 조건에서 MERT는 Handcrafted Macro-F1 0.7447보다 0.1533 높았다.
- 결과와 confusion matrix는 `results/generator_attribution/mert/`에 저장했다.

**최종 상태: MERT Generator Attribution Core QC PASS = True.**